In [1]:
import numpy as np
import pandas as pd
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
df = pd.read_parquet('data/wiki_literature_preprocessed.parquet').reset_index(drop=True)

In [3]:
tfidf = TfidfVectorizer(max_features=5000, min_df=2, max_df=0.8)
tfidf_matrix = tfidf.fit_transform(df['text_lemmatized'])

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')
dense_matrix = model.encode(
    df['summary'].tolist(),
    show_progress_bar=False,
    batch_size=64,
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
np.save("weights/dense_embeddings.npy", dense_matrix)
joblib.dump(tfidf, "weights/tfidf_model.joblib")

['tfidf_model.joblib']

In [6]:
from scraping import scrape_article
from preprocessing import preprocess_text

In [7]:
def get_article_vectors(link, debug=False):
    link_clean = link.strip()

    matching_rows = df[df['url'].str.lower() == link_clean.lower()]
    
    if matching_rows.empty:
        if debug:
            print(f"New article - scraping and processing: {link_clean}")
        article_data = scrape_article(link_clean)
        if not article_data:
            raise ValueError(f"Failed to scrape article at: {link_clean}")

        processed = preprocess_text(article_data['text'])

        tfidf_vector = tfidf.transform([processed["text_lemmatized"]]).toarray().flatten()
        
        dense_vector = model.encode(
            [article_data["summary"]],
            normalize_embeddings=True
        )[0]

        return {
            "title": article_data["title"],
            "url": link_clean,
            "dense_vector": dense_vector,
            "tfidf_vector": tfidf_vector,
            "in_database": False,
            "db_index": None
        }

    else:
        if debug:
            print(f"Article found in database: {link_clean}")
        index = matching_rows.index[0]
        
        return {
            "title": df.loc[index, 'title'],
            "url": df.loc[index, 'url'],
            "dense_vector": dense_matrix[index],
            "tfidf_vector": tfidf_matrix[index].toarray().flatten(),
            "in_database": True,
            "db_index": index
        }

In [8]:
def min_max_norm(arr):
    if (arr.max()-arr.min()) > 0:
        return (arr-arr.min()) / (arr.max()-arr.min())
    else:
        return arr

In [ ]:
def recommend_articles(visited_links, top_k, method='hybrid', alpha=0.7):
    dense_vectors = []
    tfidf_vectors = []
    excluded_indices = []

    for link in visited_links:
        try:
            article_vectors = get_article_vectors(link)
            dense_vectors.append(article_vectors["dense_vector"])
            tfidf_vectors.append(article_vectors["tfidf_vector"])

            if article_vectors["in_database"]:
                excluded_indices.append(article_vectors["db_index"])

        except Exception as e:
            print(f"Skipping link: {link}. Due to error: {e}")

    if not dense_vectors or not tfidf_vectors:
        raise ValueError("None of the links was successfully processed.")


    user_dense_profile = np.mean(dense_vectors, axis=0)
    user_dense_profile_norm = np.linalg.norm(user_dense_profile)
    if user_dense_profile_norm > 0:
        user_dense_profile = user_dense_profile / user_dense_profile_norm

    user_tfidf_profile = np.mean(tfidf_vectors, axis=0, keepdims=True)


    dense_similarities = np.dot(dense_matrix, user_dense_profile)
    tfidf_similarities = cosine_similarity(tfidf_matrix, user_tfidf_profile).flatten()

    match method:
        case 'dense':
            final_similarities = dense_similarities
        case 'tfidf':
            final_similarities = tfidf_similarities
        case 'hybrid':
            final_similarities = (alpha * min_max_norm(dense_similarities)) + ((1-alpha) * min_max_norm(tfidf_similarities))
        case _:
            raise ValueError("Invalid method. Choose from: 'dense', 'tfidf', 'hybrid'.")

    
    results_df = df.copy()
    results_df['final_score'] = final_similarities
    results_df['dense_score'] = dense_similarities
    results_df['tfidf_score'] = tfidf_similarities

    if excluded_indices:
        results_df = results_df.drop(index=excluded_indices)

    top_recommendations = results_df.nlargest(top_k, 'final_score')

    return top_recommendations[['title', 'url', 'final_score', 'dense_score', 'tfidf_score', 'summary']]

## tests

In [10]:
# user_input_links = [
#     "https://en.wikipedia.org/wiki/Italo_Calvino"
# ]

# recs = recommend_articles(user_input_links, top_k=5, method="hybrid")
# display(recs[["title", "final_score", 'dense_score', 'tfidf_score', "url"]])

In [11]:
# # unseen article test
# user_input_links = [
#     "https://en.wikipedia.org/wiki/Invisible_Cities"
# ]

# recs = recommend_articles(user_input_links, top_k=5, method="hybrid")
# display(recs[["title", "final_score", 'dense_score', 'tfidf_score', "url"]])

In [12]:
# recs = recommend_articles(user_input_links, top_k=2, method="dense")
# display(recs[["title", "final_score", 'dense_score', 'tfidf_score', "url"]])

In [13]:
# recs = recommend_articles(user_input_links, top_k=2, method="tfidf")
# display(recs[["title", "final_score", 'dense_score', 'tfidf_score', "url"]])

In [14]:
# # already seen article test
# user_input_links = [
#     "https://en.wikipedia.org/wiki/Jean-Paul_Sartre"
# ]

# recs = recommend_articles(user_input_links, top_k=5, method="hybrid")
# display(recs[["title", "final_score", 'dense_score', 'tfidf_score', "url"]])

In [15]:
# # mix of seen and unseen articles
# user_input_links = [
#     "https://en.wikipedia.org/wiki/Jean-Paul_Sartre",
#     "https://en.wikipedia.org/wiki/Albert_Camus",
    
#     "https://en.wikipedia.org/wiki/Italo_Calvino",
#     "https://en.wikipedia.org/wiki/Invisible_Cities",
# ]

# recs = recommend_articles(user_input_links, top_k=5, method="hybrid")
# display(recs[["title", "final_score", 'dense_score', 'tfidf_score', "url"]])